# Agent-based Model of Sign Language Emergence

*Written by Katie Mudd*\
*Updated by Marieke Schouwstra, Caspar Fermin and Francijn Keur*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/evolamus/labs/blob/main/Agent_based_model_sign_language.ipynb?py=r)

**Don’t forget to make a copy of the notebook in your own Drive to save your changes!**

In this computer lab you will partially recreate an agent-based model of sign language emergence based on an article by [Mudd, de Vos and de Boer (2022)]( https://doi.org/10.3390/languages7010031). As a reminder, this model is about how shared context affects lexical variation. First, we will first make some agents (i.e. individuals in the model). Second, we will output the results from the language game. Third, we will calculate lexical variation between agents. Fourth, we will compare these results to another experiment. A part of this model - the language game - is code that you do not need to understand line by line; understanding the basic idea is the purpose of this assignment.

The lab contains <font color="orange"><b>Do</b></font> questions, where you need to run some code,or calculate something yourself, and <font color="green"><b>Think</b></font> questions, which check your understanding of what is happening.

Some notes on using this notebook: 

- You should run all the code cells in this notebook, either by clicking the 'play button' that appears once you hover over them (in Google Colab) or by pressing Shift+Enter from inside each cell. Make sure you don't miss any! Once a cell has completed running, a number will appear between the [square brackets] to the left of the cell.
- You may run a cell multiple times (for example to check how the output changes). 
- In principle, you could also run a previous cell again after you've run the cells below it. However, this might be a bad idea, and cause you trouble once you try to run new code further down the line again. If this happens and you run into strange errors, you can always try to restart the notebook (Ctrl+M or _Runtime > Restart runtime_ in the menu) and carefully run each cell from top to bottom again.
- To see an overview of the entire computer lab, and easily navigate between sections, open the _Table of contents_ tab to the left (first icon in the navigation bar).

## Background

During one of the lectures, you were shown examples of _lexical variation_ and _iconicity_ in English and in a few sign languages.

>**Exercise 1**
>
> <font color="green"><b>Think:</b></font>
>   * Lexical variation refers to the existence of different forms to refer roughly to the same concept. Provide at least one example of lexical variation in any language (other than the examples provided in the lecture and the ones that we came up with in the lecture). Which social groups (e.g., age, gender, region) use the different variants?
>   * Iconicity refers to the resemblance between a form and a meaning. Provide at least one example of iconicity (again in any language of your choice; as long as it is not an example discussed in class). In what way do you find this form to be iconic?

>**_Fill in your answer here_**


Because iconicity is subjective and depends on one's experience, __shared context__ (i.e., shared social and psychological information) is required for iconic form-meaning mappings to make sense for people who are communicating.


## About the model

A bit about the different agent attributes:

* __agent_id__ a number assigned to identify individual agents (an integer; e.g., 7)
* __group__ the group an agent belongs to, determines their csf (an integer;  e.g., 2)
* __language_concept__ a letter denoting individual concepts (a string; e.g., B)
* __language_csf__ the "culturally salient features" assigned per concept based on group (a string of 0s and 1s; e.g., 010) 
* __language_form__ the form generated based on the intial_degree_of_overlap with language_csf (a string of 0s and 1s; e.g., 001)

Model input parameters:

* __n_groups__ number of groups which determine the "culturally salient features" of agents (__language_csf1__). E.g., 2
* __n_bits__ the number of bits determining the length of __language_csf1__ and __language_form1__. E.g., 3
* __intial_degree_of_overlap__ the probability that each bit of __language_csf1__ will match that of __language_form1__. E.g., 0.6

More concretely, if the __intial_degree_of_overlap__ = 0.6, for each bit of __language_csf1__ there is a 60% chance that the bit of the form will be identical to that of the __language_csf1__ . In the other 40% of cases, a 0 or 1 is randomly assigned. 

The code below creates an agent who knows two concepts. The initial degree of overlap is high.

In [176]:
# run this first to install and load a package!

install.packages('gdata')
install.packages('DescTools')
install.packages('dplyr')

library(gdata)
library(DescTools)
library(dplyr)

Warning message:
"package 'gdata' is in use and will not be installed"
Warning message:
"package 'DescTools' is in use and will not be installed"
Warning message:
"package 'dplyr' is in use and will not be installed"


In [177]:
agent1_conceptA <- data.frame( agent_id = 1,
                               group = 1,
                               language_concept = "A",
                               language_csf = "11111",
                               language_form = "11011",
                               stringsAsFactors = FALSE)  # otherwise R produces errors when changing the df

agent1_conceptB <- data.frame( agent_id = 1,
                               group = 1,
                               language_concept = "B",
                               language_csf = "00000",
                               language_form = "10000",
                               stringsAsFactors = FALSE)

agent1 <- rbind(agent1_conceptA, agent1_conceptB)  # rbind puts the dataframes together
agent1

agent_id,group,language_concept,language_csf,language_form
<dbl>,<dbl>,<chr>,<chr>,<chr>
1,1,A,11111,11011
1,1,B,00000,10000


Now let's make another agent who belongs to a different cultural group (group 2 instead of 1), and there is no shared context between the two agents.

In [178]:
agent2_conceptA <- data.frame( agent_id = 2,
                               group = 2,
                               language_concept = "A",
                               language_csf = "00111",
                               language_form = "00010",
                               stringsAsFactors = FALSE)

agent2_conceptB <- data.frame( agent_id = 2,
                               group = 2,
                               language_concept = "B",
                               language_csf = "11100",
                               language_form = "11101",
                               stringsAsFactors = FALSE)

agent2 <- rbind(agent2_conceptA, agent2_conceptB)
agent2

agent_id,group,language_concept,language_csf,language_form
<dbl>,<dbl>,<chr>,<chr>,<chr>
2,2,A,00111,00010
2,2,B,11100,11101


>**Exercise 2: Creating an agent with two concepts**
>
><font color="green"><b>Think:</b></font>
>   * Explain, using the code for agent 1 and 2, how you can see that they are from two different cultural groups. Which part of the code, apart from _group_, is relevant here?
>   * Above we specified that initial degree of overlap is high for these agents. Observe the __language_forms__ for __language_concept__ "A" and "B" and specify what you think the value of __intial_degree_of_overlap__ might be. Assign your chosen value to the variable _initial_overlap_.

>**_Fill in your answers here_**

In [ ]:
initial_overlap <-  # fill in your value here!

Now we put the two agents together:

In [180]:
agents_diff_groups <- rbind (agent1, agent2)
agents_diff_groups

agent_id,group,language_concept,language_csf,language_form
<dbl>,<dbl>,<chr>,<chr>,<chr>
1,1,A,11111,11011
1,1,B,00000,10000
2,2,A,00111,00010
2,2,B,11100,11101


Now, we would like to add a few more concepts to the dataframe, but doing this all manually is a bit tedious. Therefore, we created a function "add_more_concepts" that helps us generate more concepts, which takes as input parameters _agents_, _agent_id_, _group_, _n_concepts_, _n_bits_ and _initial_overlap_. You do not need to understand this code line by line.

In [181]:
add_more_concepts <- function(agents, id, group, n_concepts, n_bits, initial_overlap){

    # repeating the code for the number of concepts that you want to add
    for (i in 1:n_concepts){
    
        # obtaining the next character to use for a language_concept, for this agent 
        agent_df <- subset(agents, agent_id == id)
        concept <- AscToChar(CharToAsc(max(agent_df$language_concept)) + 1 %% 26) 

        # first we check if there is already a csf for this group and concept in the dataframe 
        existing_csf_concept <- agents[agents$group == group & agents$language_concept == concept,]


        if (nrow(existing_csf_concept) > 0){
            csf <- existing_csf_concept$language_csf
            # change it to a vector instead of a string
            csf <- as.numeric(strsplit(csf, split = "")[[1]])
        }
        # if there is no csf yet for this group and concept, we create a random vector of length n_bits with 0s and 1s.
        else{

            existing_csfs <- agents[agents$group == group,]$language_csf
            csf <- NULL

            # making sure that the csf is unique for this group (i.e. two concepts cannot have the same csf string)
            while (is.null(csf) || paste(csf, collapse = "") %in% existing_csfs){
                csf <- rbinom(n=n_bits, size=1, prob=runif(1, 0, 1))
                
            }
        }

        # to decide whether the language concept bit is the same as the csf we use the match_csf vector
        match_csf <- rbinom(n = n_bits, size = 1, prob = initial_overlap)
      
        # in case the bit should not match, we need a random bit
        random_bits <- rbinom(n = n_bits, size = 1, prob = 0.5)

        # now we check for every bit if it should stay the same, and otherwise we get a random bit
        form <- ifelse(match_csf == 1, csf, random_bits)

        # changing the vectors into strings
        csf <- paste(csf, collapse = "")
        form <- paste(form, collapse = "")

        new_concept <- data.frame(
                        agent_id = id,
                        group = group,
                        language_concept = concept,
                        language_csf = csf,
                        language_form = form,
                        stringsAsFactors = FALSE
        )

        agents <- rbind(agents, new_concept)

        
    }
    return(agents)
}

# adding concepts for agent 1
agents_diff_groups <- add_more_concepts(agents=agents_diff_groups, id=1, group=1, n_concepts=5, n_bits=5, initial_overlap = initial_overlap)

# and for agent 2
agents_diff_groups <- add_more_concepts(agents=agents_diff_groups, id=2, group=2, n_concepts=5, n_bits=5, initial_overlap = initial_overlap)

agents_diff_groups

agent_id,group,language_concept,language_csf,language_form
<dbl>,<dbl>,<chr>,<chr>,<chr>
1,1,A,11111,11011
1,1,B,00000,10000
2,2,A,00111,00010
2,2,B,11100,11101
1,1,C,00100,00100
1,1,D,00010,10010
1,1,E,11010,11010
1,1,F,10111,10111
1,1,G,10000,11100


Here is the code for the language game part of the model. You do not need to understand this code line by line, but you should understand *conceptually* what is being done for the purpose of this assignment.

##### 1. Stop after form-form match: compare the comprehender's closest form to the producer's form (conventional link)


In [182]:
does_closest_form_match <- function(agents, producer_id, producer_concept_choice, comprehender_id){
  
  # get the producer's form
  producer <- subset(agents, agent_id == producer_id)
  producer_concept_row <- subset(producer, language_concept == producer_concept_choice)
  producer_form <- producer_concept_row$language_form
  
  # calculate the distance between the producer's form and all the comprehender's forms
  comprehender <- subset(agents, agent_id == comprehender_id)
  
  concept_distance_df <- data.frame()  # empty dataframe to add concept and distance pairs
  
  for (i in 1:nrow(comprehender)){  # iterate through the comprehender's rows
    comprehender_row <- comprehender[i,]
    comprehender_form <- comprehender_row$language_form

    compare_forms <- apply(do.call(rbind, strsplit(c(as.character(producer_form), as.character(comprehender_form)), "")), 2, function(x){length(unique(x)) == 1})
    
    number_of_bits <- length(compare_forms)
    number_of_matches <- sum(compare_forms, na.rm = TRUE)
    forms_distance <- number_of_bits - number_of_matches
    
    comprehender_concept <- comprehender_row$language_concept

    add_concept_distance <- data.frame(concept = comprehender_concept, 
                                       distance = forms_distance)
    concept_distance_df <- rbind(concept_distance_df, add_concept_distance)
    
  # find the comprehender's closest form to the producer's
  min_concept <- with(concept_distance_df, concept[distance == min(distance)])  # returns the concept with minimum distance

  # if multiple with minimum distance (i.e., there is a tie), randomly choose one
  if (length(min_concept) > 1){
    min_concept <- sample(min_concept, 1)
    }
    
}
  return(min_concept == producer_concept_choice)
}

#### 2. Stop after csf-form match: compare the comprehender's closest culturally salient features (CSF) to the producer's form

In [183]:
does_closest_csf_match <- function(agents, producer_id, producer_concept_choice, comprehender_id){
  
  # get the producer's form
  producer <- subset(agents, agent_id == producer_id)
  producer_concept_row <- subset(producer, language_concept == producer_concept_choice)
  producer_form <- producer_concept_row$language_form
  
  # calculate distance between the producer's form and all the comprehender's CSF
  comprehender <- subset(agents, agent_id == comprehender_id)
  
  concept_distance_df <- data.frame()  # empty dataframe to add concept and distance pairs
  
  for (i in 1:nrow(comprehender)){  # iterate through the the comprehender's rows
    comprehender_row <- comprehender[i,]
    comprehender_csf <- comprehender_row$language_csf

    compare_bits <- apply(do.call(rbind, strsplit(c(as.character(producer_form), as.character(comprehender_csf)), "")), 2, function(x){length(unique(x)) == 1})
    
    number_of_bits <- length(compare_bits)
    number_of_matches <- sum(compare_bits, na.rm = TRUE)
    bits_distance <- number_of_bits - number_of_matches
    
    comprehender_concept <- comprehender_row$language_concept

    add_concept_distance <- data.frame(concept = comprehender_concept, 
                                       distance = bits_distance)
    concept_distance_df <- rbind(concept_distance_df, add_concept_distance)
    
  # find the comprehender's closest CSF to the producer's form
  min_concept <- with(concept_distance_df, concept[distance == min(distance)])  # returns the comprehender's concept for their CSF with minimum distance to producer's form

  # if multiple with minimum distance (i.e., there is a tie), randomly choose one
  if (length(min_concept) > 1){
    min_concept <- sample(min_concept, 1)
    }
    
}
  return(min_concept == producer_concept_choice)
}


##### 3. Update the comprehender's form

In [184]:
update_comprehender_form <- function(agents, producer_id, producer_concept_choice, comprehender_id){

  # get the producer's form
  producer <- subset(agents, agent_id == producer_id)
  producer_concept_row <- subset(producer, language_concept == producer_concept_choice)
  producer_form <- producer_concept_row$language_form

  # get the comprehender's form
  comprehender <- subset(agents, agent_id == comprehender_id)
  comprehender_concept <- subset(comprehender, language_concept == producer_concept_choice)
  comprehender_form <- comprehender_concept$language_form

  # compare the producer's and comprehender's forms
  compare_forms <- apply(do.call(rbind, strsplit(c(as.character(producer_form), as.character(comprehender_form)), "")), 2, function(x){length(unique(x)) == 1})
  
  # find bits that don't match between the producer's and the comprehender's forms
  indexes_not_matching <- which(compare_forms == FALSE)
  
  if (length(indexes_not_matching) > 0){
    print(paste0("non-matching index: ",indexes_not_matching))
    index_to_flip <- gdata::resample(indexes_not_matching, 1)
    
    print(paste0("Now changing index ",index_to_flip))
    comprehender_form_split <- strsplit(as.character(comprehender_form), split="")[[1]]
    bit_to_flip <- comprehender_form_split[index_to_flip]
    flipped_bit <- (1-as.numeric(bit_to_flip))
    
    form_to_update <- as.character(comprehender_form)
    substr(form_to_update, index_to_flip, index_to_flip) <- as.character(flipped_bit)
    
    # update the comprehender's form in the dataframe
    agents$language_form[agents$agent_id == comprehender_id & agents$language_concept == producer_concept_choice] <- form_to_update
    return(agents)
  }
  else{
    print("There is another concept with the same form which got picked, not changing any indexes.")
    return(agents)
  }
}

There are three possible ways in which the language game can end (after 1, after 2 or after 3).:

1. Stop after form-form match: compare the comprehender's closest form to the producer's form _(conventional link)_. If the concepts corresponding to the forms match then the language game stops here.
2. Stop after csf-form match: compare the comprehender's closest culturally salient features (CSF) to the producer's form _(iconic-inferential pathway)_. If the concept corresponding to the comprehender's CSF and the producer's form match then the language game stops here.
3. Update the comprehender's form: for the concept chosen by the producer, change one bit of the comprehender’s form to be the same as the producer's form.


>**Exercise 3a: Running the language game code**
>
><font color="green"><b>Think:</b></font>
>   * Now that there are two agents in the model with two concepts each, reflect on which cases would lead to the language game stopping after step 1, the language game stopping after step 2 and the language game ending at step 3 with updating the comprehender's form. Use examples of each (if possible) from the agents. 

>**_Fill in your answer here_**

In the next codeblock "Selecting agents to be the producer and comprehender and running the language game", agents from the agent list you have just made are selected to play the language game. A producer and a comprehender are selected from the agent list, and then the agents go through the steps of the language game. If the language game stops at step 1 or 2, no change is made. If the language game proceeds to step 3, the comprehender's *form* gets updated. The language game is played for _n_rounds_.


##### Selecting agents to be the producer and comprehender and running the language game

In [185]:
runsimulation <- function(agents_df, n_rounds){

agent_ids <- unique(agents_df$agent_id)

for (round in 1:n_rounds){          
  for (agent in agent_ids){
    # choose a producer
    producer <- agent
    print(paste0("producer: ", producer))

    # choose a concept
    producer_concept_choice <- sample(unique(agents_df$language_concept), 1)
    print(paste0("producer concept choice: ", producer_concept_choice))

    # choose a comprehender
    ids <- agents_df[,1]# get ids
    comprehender <- sample(ids[ids != agent], 1)  # choose an id randomly to be the comprehender (this should not be the producer)

      if (does_closest_form_match(agents_df, producer, producer_concept_choice, comprehender) == TRUE) {
      print('Stop at step 1: producer form and comprehender form match')
      }  else if (does_closest_csf_match(agents_df, producer, producer_concept_choice, comprehender) == TRUE) {
      print('Stop at step 2: producer form and comprehender csf match')
      } else {
      agents_df <- update_comprehender_form(agents_df, producer, producer_concept_choice, comprehender)
      print('Step 3: updated comprehender bit')
      }
  
    print(agents_df)
  }
} 

return(agents_df)
}

In [186]:
# sorting the dataframe by concepts to make comparing forms a bit easier:
agents_diff_groups <- arrange(agents_diff_groups, language_concept) 

In [ ]:
# run this to run the simulation for agents with different groups!
agents_diff_groups_updated <- runsimulation(agents_diff_groups, 10)

[1] "producer: 1"
[1] "producer concept choice: G"
[1] "non-matching index: 4" "non-matching index: 5"
[1] "Now changing index 4"
[1] "Step 3: updated comprehender bit"
   agent_id group language_concept language_csf language_form
1         1     1                A        11111         11011
2         2     2                A        00111         00010
3         1     1                B        00000         10000
4         2     2                B        11100         11101
5         1     1                C        00100         00100
6         2     2                C        00001         00001
7         1     1                D        00010         10010
8         2     2                D        11101         11101
9         1     1                E        11010         11010
10        2     2                E        00100         00000
11        1     1                F        10111         10111
12        2     2                F        11110         11010
13        1     1        

>**Exercise 3b: Running the language game code**
>
><font color="green"><b>Think:</b></font>
>   * Each time the language game is played, several things are printed. What do they refer to? You will need to look into the code for the answer.

>**_Fill in your answer here_**

For each pair of agents in the population the lexical distance is calculated by comparing forms for each concept and then taking the average. For each form that describes a concept, the distance is 1 if at least one of the bits is different and the distance is 0 if the bits are the same. 

In [188]:
agents_diff_groups

agent_id,group,language_concept,language_csf,language_form
<dbl>,<dbl>,<chr>,<chr>,<chr>
1,1,A,11111,11011
2,2,A,00111,00010
1,1,B,00000,10000
2,2,B,11100,11101
1,1,C,00100,00100
2,2,C,00001,00001
1,1,D,00010,10010
2,2,D,11101,11101
1,1,E,11010,11010


>**Exercise 4: Calculating lexical distances between agents**
>
><font color="orange"><b>Do:</b></font>
>
> Calculate the average lexical distance between the two agents by hand.

>**_Fill in your answer here_**

Here is some code to calculate the average lexical distances between the two agents:

In [189]:
# this code is just for when there are 2 agents in the population. If you'd make a larger simulation, you can work with multiple pairwise comparisons, but you'd need additional code. (Figuring out what this code should be is a fun exercise if you're interested in coding!)

calculate_average_distance <- function(agents_df){
  concept_distance_df <- data.frame()  # empty dataframe to add concept and distance pairs
  for (concept in unique(agents_df$language_concept)){
    agent1_concept <- subset(agents_df, agent_id == 1 & language_concept == concept)
    agent2_concept <- subset(agents_df, agent_id == 2 & language_concept == concept)

    compare_form_bits <- apply(do.call(rbind, strsplit(c(as.character(agent1_concept$language_form), as.character(agent2_concept$language_form)), "")), 2, function(x){length(unique(x)) == 1})
      
    number_of_bits <- length(compare_form_bits)
    number_of_matches <- sum(compare_form_bits, na.rm = TRUE)
    bits_distance <- number_of_bits - number_of_matches
    if (bits_distance > 0) {
    add_concept_distance <- data.frame(language_concept = concept, distance = 1) # forms are different
} else {
    add_concept_distance <- data.frame(language_concept = concept, distance = 0)  # forms match
}
    
    concept_distance_df <- rbind(concept_distance_df, add_concept_distance)

  }
  return(mean(concept_distance_df$distance))  # caculate average of all distances
  
}

In [227]:
# run this to run the simulation for agents with different groups!
agents_diff_groups_updated <- runsimulation(agents_diff_groups, n_rounds = 10)

[1] "producer: 1"
[1] "producer concept choice: D"
[1] "non-matching index: 2" "non-matching index: 3" "non-matching index: 4"
[4] "non-matching index: 5"
[1] "Now changing index 2"
[1] "Step 3: updated comprehender bit"
   agent_id group language_concept language_csf language_form
1         1     1                A        11111         11011
2         2     2                A        00111         00010
3         1     1                B        00000         10000
4         2     2                B        11100         11101
5         1     1                C        00100         00100
6         2     2                C        00001         00001
7         1     1                D        00010         10010
8         2     2                D        11101         10101
9         1     1                E        11010         11010
10        2     2                E        00100         00000
11        1     1                F        10111         10111
12        2     2                F 

In [228]:
print(paste0("Diff-groups-simulation initial average distance: ",calculate_average_distance(agents_diff_groups)))

print(paste0("Diff-groups-simulation average distance after interaction: ",calculate_average_distance(agents_diff_groups_updated)))

[1] "Diff-groups-simulation initial average distance: 1"
[1] "Diff-groups-simulation average distance after interaction: 0.428571428571429"


Now let's compare this result to the case where all the agents belong to the same group. In this case all the culturally salient features of agents are the same. As done before, let's make some agents!

In [203]:
agent1_conceptA_samegroup <- data.frame( agent_id = 1,
                               group = 1,
                               language_concept = "A",
                               language_csf = "00111",
                               language_form = "00110",
                               stringsAsFactors = FALSE)

agent1_conceptB_samegroup <- data.frame( agent_id = 1,
                               group = 1,
                               language_concept = "B",
                               language_csf = "11111",
                               language_form = "01111",
                               stringsAsFactors = FALSE)

agent2_conceptA_samegroup <- data.frame( agent_id = 2,
                               group = 1,
                               language_concept = "A",
                               language_csf = "00111",
                               language_form = "00111",
                               stringsAsFactors = FALSE)

agent2_conceptB_samegroup <- data.frame( agent_id = 2,
                               group = 1,
                               language_concept = "B",
                               language_csf = "11111",
                               language_form = "01110",
                               stringsAsFactors = FALSE)

#overwrite variable 'agents' with the new same-group agents 
agents_samegroup <- rbind(agent1_conceptA_samegroup, agent1_conceptB_samegroup, agent2_conceptA_samegroup, agent2_conceptB_samegroup)  # rbind puts the dataframes together

# adding more concepts for agent 1
agents_samegroup <- add_more_concepts(agents_samegroup, id = 1, group = 1, n_concepts = 5, n_bits = 5, initial_overlap = initial_overlap)

# and for agent 2
agents_samegroup <- add_more_concepts(agents_samegroup, id = 2, group = 1, n_concepts = 5, n_bits = 5, initial_overlap = initial_overlap)

agents_samegroup <- arrange(agents_samegroup, language_concept)

agents_samegroup

agent_id,group,language_concept,language_csf,language_form
<dbl>,<dbl>,<chr>,<chr>,<chr>
1,1,A,00111,00110
2,1,A,00111,00111
1,1,B,11111,01111
2,1,B,11111,01110
1,1,C,00000,00010
2,1,C,00000,01001
1,1,D,00110,00111
2,1,D,00110,01010
1,1,E,10111,00101


In [217]:
# run this to run the simulation!
agents_samegroup_updated <- runsimulation(agents_samegroup, 10)

[1] "producer: 1"
[1] "producer concept choice: D"
[1] "non-matching index: 2" "non-matching index: 3" "non-matching index: 5"
[1] "Now changing index 2"
[1] "Step 3: updated comprehender bit"
   agent_id group language_concept language_csf language_form
1         1     1                A        00111         00110
2         2     1                A        00111         00111
3         1     1                B        11111         01111
4         2     1                B        11111         01110
5         1     1                C        00000         00010
6         2     1                C        00000         01001
7         1     1                D        00110         00111
8         2     1                D        00110         00010
9         1     1                E        10111         00101
10        2     1                E        10111         10111
11        1     1                F        01010         11010
12        2     1                F        01010         01011
1

In [218]:
print(paste0("Same-group simulation initial average distance: ",calculate_average_distance(agents_samegroup)))
print(paste0("Same-group simulation average distance after communication: ",calculate_average_distance(agents_samegroup_updated)))

[1] "Same-group simulation initial average distance: 1"
[1] "Same-group simulation average distance after communication: 0.714285714285714"


>**Exercise 5a: Comparing the results to another experiment**
>
><font color="orange"><b>Do:</b></font>
>   * Compare the result of the lexical distance when the two agents are in the same group and when the two agents are in different groups. Run the simulations at least 5 times, with _n_rounds_ = 10; report and compare the outcomes. Does every run of the model produce what you expected? What happens if you increase the value of _initial_overlap_?

>**_fill in your answer here_**

>**Exercise 5b: Comparing the results to another experiment**
>
><font color="green"><b>Think:</b></font>
>   * If the agents in the two different experiments continued playing the language game (i.e. _n_rounds_ is increased), do you think the result would stay the same? Or do you think the lexical distances would be higher between the agents in the first experiment (agents in different groups - i.e., no shared context) or between the agents in the second experiment (agents in the same group - i.e., shared context). Motivate your answer. 

>**_fill in your answer here_**

>**Exercise 5c: Comparing the results to another experiment**
>
><font color="green"><b>Think:</b></font>
> * Finally we'd like you to reflect on how exactly shared context, lexical variation, and agent communication are modeled. Specify two ways in which the model simplifies the real world, and evaluate this simplification: would there have been different (better?) ways to model it? 


>**_fill in your answer here_**